# BPE

## Abstract

NMT모델은 일반적으로 고정된 크기의 단어 범위 안에서 수행됐지만 번역은 새로운 단어, 합성어 등이 등장하는 open-vocabulary 문제이다. 이 문제를 이전에는 사전 조회 방식으로 처리했지만(backing off to a dictionary) 이 논문에서는 자주 안쓰이거나 모르는 단어를 subword unit들의 연결로 처리하는 방식을 제안한다.

## Introduction
Neural Model의 단어는 보통 30000 ~ 50000개의 단어들로 제한 되어있지만 번역은 open-vocabulary problem 이다. 특히 교착어나 합성어처럼 단어의 생성이 활발하다면 언어모델은 단어보다 더 작은 수준에서의 메커니즘을 필요로한다. 

예시 : 하수 처리 시설 이란 단어는 '하수처리시설' 로 처리하는 것보다 '하수' '처리' '시설' 와 같이 직관적으로 처리하는 것이 더 좋다.

단어 단위 NMT 모델에서 번역에서의 out-of-vocabulary(OOV) 단어는 back-off to a dictionary look up을 통해 다뤄진다. 하지만 이러한 방식은 모든 단어가 타겟과 1대1 매칭이 되는 것도 아니고 학습과정에서 배우지 못한 단어를 생성하거나 번역하는 것은 불가능하다.

이 논문에서는 sub-word 기반 모델이 번역 과정을 단순하게 만드는 것과 함께 희귀 단어 번역에 대해 더 높은 정확도를 이끌어내고 훈련 시점에 한번도 보지 못한 단어를 효울적으로 생성하는 것을 발견했다.

이 논문은 두가지 기여를 한다.
1. 단어를 서브워드 단위로 인코딩해서 open vocabulary에 대한 Neural Machine 번역이 가능함을 보인다.
2. 압축 알고리즘인 byte pair encoding(BPE)를 단어 segmentation과정에 적용하며 open-vocabulary의 표현을 가변 길이 문자 sequence를 기반으로 어휘 사전을 통해 표현할 수 있게한다.

## Neural Machine Translation

이 논문에서는 Bahdanau et al.(2015) 의 neural machine translation 구조를 따른다.  

### NMT란?
neural machine translation 은 순환신경망(RNN)을 기반으로한 encoder-decoder 구조로 구현된다.  

인코더는 GRU로 이루어진 양방향 인공 신경망이다. Input sequence $x = [x_1, x_2, ..., x_m]$ 을 읽어 forward sequence $\overrightarrow{h_i}$ , backward sequece $\overleftarrow{h_i}$를 계산한다. 그리고 각 hidden state를 concatenate해서 annotation vector $h_j$를 만들게 된다.   

디코더는 target sequence $y = (y_1, y_2, ... , y_n)$을 예측하는 RNN 모델이다. 각 단어 $y_i$는 순환 은닉상태 $s_i$, 바로 이전 예측인 $y_{i-1}$, 문맥 벡터(context vector) $c_i$로 구성된다. $c_i$는 $h_j$의 가중합으로 계산된다. 각 $h_j$는 $\alpha_{ij}$로 부터 계산되며 $\alpha_{ij}$는 단어 $y_i$가 입력 $x_j$에 의해 정렬될 확률을 모델링한다. 이 정렬 모델은 레이어가 하나인 feedforward neural network이고 역전파를 통해 나머지 네트워크와 함께 학습된다.


## Subword Translation
이 논문의 핵심 동기는 몇몇 단어들은 번역이 쉽다는 것이다. 몇몇 단어는 새롭더라도 형태소나 음소와 같은 subword를 기반으로 번역이 가능하다.  
1) 같은 알파벳을 사용한다면 source 언어에서 target 언어로 그대로 복사가 가능하다. 하지만 사용하는 알파벳이 다르다면 전사(transcription) 또는 음역(transliteration)이 필요하게 된다.  
ex.  
Barack Obama (English; German)  
Барак Обама (Russian)  
バラク・オバマ (ba-ra-ku o-ba-ma) (Japanese)  

2) 동원어(cognates) 또는 차용어(loanword)같은 경우 언어간 비슷한 차이가 있어 글자 수준의 번역 규칙으로도 충분하다.  
ex.  
claustrophobia (English)  
Klaustrophobie (German)  
Клаустрофобия (Klaustrofobiâ) (Russian)  

3) 형태적으로 복잡한 단어들은 다양한 형태소를 포함하고있어 각 형태소를 번역하면 된다.  
ex.  
solar system (English)  
Sonnensystem (Sonne + System) (German)  
Naprendszer (Nap + Rendszer) (Hungarian)  

실제로 독일어 학습 데이터중 100개의 희귀 토큰 분석에서 대다수의 토큰은 더 작은 단위를 통해 영어로 번역 가능함을 알 수 있었다.

### Related work

통계적 기계 번역(Statistical Machine Translation, SMT)는 Unknown word에 대한 다양한 번역 문제가 연구되어 왔다. 문자 기반 번역은 구문 기반 번역과 함께 연구되어 비슷한 언어 사이에서 효과적임을 알 수 있었고
특히 SMT에 널리 사용되는 다양한 형태소분할 알고리즘이 연구되었다. 구문기반의 SMT에 사용되는 분할 알고리즘은 분할 결정에 있어 보수적인 경향이 있지만 이 논문에서는 공격적인 분할을 통해 간결한 어휘 네트워크를 만들고 back-off dictionaries 없이 open-vocabulary 번역을 가능하게 한다.  

다양한 subword unit을 선택하는 방식을 NMT에 적용하려는 시도가 존재했지만 유의미한 향상을 이끌어내지는 못했다. 그 이유는 단어수준에서 어텐션 매커니즘이 작동하고 이러한 각 단어에 대한 표현은 고정된 길이였기 때문이라고 판단하였다. 논문에서는 가변 길이 표현을 통해 어텐션 메커니즘을 각 스텝에서 다른 subword unit에 대해 가중치를 두는 것이 이점을 가져올 것이라고 하였고 긴 복합어를 하나의 고정길이 벡터로 활용하는 정보 병목이 줄어들 것이라고 판단했다.

> 고정 길이 벡터(fixed-length continuous word vectors) : 위의 '하수처리시설'의 예시처럼 단어 하나를 하나의 벡터로 표현함  

NMT는 구문기반의 방식과 달리 모델의 vocabulary 크기를 최소화하여 시간, 메모리효율을 높이고 back-off모델 없이도 번역이 가능하게 한다. 동시에 텍스트 자체도 간결하게 표현되어야 하는데 텍스트 길이가 길어지면 효율성이 떨어지고 정보 전달의 거리가 증가하기 때문이다. 이러한 단어 크기와 텍스트 크기의 trade off를 조절하기 위해 분할되지 않는 단어는 간단한 목록으로 저장하고 rare-word에 대해서만 subword unit을 사용하는 것이다. 이에 대한 대안으로 BPE기반의 알고리즘을 제안한다.  

### Byte Pair Encoding(BPE)

BPE는 반복적으로 가장 많이 나오는 byte쌍을 다른 byte로 바꿔주는 간단한 압축 알고리즘
본 논문에서는 문자 또는 문자 시퀀스를 병합하며 알고리즘을 적용한다.

In [2]:
import re, collections

def get_stats(vocab):
    pairs = collections.defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pairs[symbols[i],symbols[i+1]] += freq
    return pairs

def merge_vocab(pair, v_in):
    v_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in v_in:
        w_out = p.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

vocab = {
    'l o w </w>' : 5, 
    'l o w e r </w>' : 2,
    'n e w e s t </w>':6, 
    'w i d e s t </w>':3
    }

num_merges = 10

for i in range(num_merges):
    pairs = get_stats(vocab)
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    print(vocab)

{'l o w </w>': 5, 'l o w e r </w>': 2, 'n e w es t </w>': 6, 'w i d es t </w>': 3}
{'l o w </w>': 5, 'l o w e r </w>': 2, 'n e w est </w>': 6, 'w i d est </w>': 3}
{'l o w </w>': 5, 'l o w e r </w>': 2, 'n e w est</w>': 6, 'w i d est</w>': 3}
{'lo w </w>': 5, 'lo w e r </w>': 2, 'n e w est</w>': 6, 'w i d est</w>': 3}
{'low </w>': 5, 'low e r </w>': 2, 'n e w est</w>': 6, 'w i d est</w>': 3}
{'low </w>': 5, 'low e r </w>': 2, 'ne w est</w>': 6, 'w i d est</w>': 3}
{'low </w>': 5, 'low e r </w>': 2, 'new est</w>': 6, 'w i d est</w>': 3}
{'low </w>': 5, 'low e r </w>': 2, 'newest</w>': 6, 'w i d est</w>': 3}
{'low</w>': 5, 'low e r </w>': 2, 'newest</w>': 6, 'w i d est</w>': 3}
{'low</w>': 5, 'low e r </w>': 2, 'newest</w>': 6, 'wi d est</w>': 3}


1. 단어를 character 단위로 쪼개고 끝에  ' . '를 붙인다. 이 ' . '은 번역 이후 원래 토큰을 복원할 수 있게 해준다.
2. 가장 빈번하게 발생하는 symbol pair를 새로운 symbol로 치환 ('A', 'B') -> (AB)
3. 이러한 merge로 생성된 symbol은 새로운 n-gram을 표현하게 된다.
4. 결과적으로 빈번한 문자들, 또는 전체 단어가 single symbol로 합쳐지면서 shortlist를 줄일 수 있다.
이러한 방식에서 최종 vocab 크기 는 초기 vocab (글자 개수) 과 merge 횟수의 합으로 계산된다.  
즉 merge횟수가 관리해야하는 유일한 하이퍼파라미터이며 vocab & sequence trade off를 위해 merge횟수만 정하면 된다.  

특히 효율성을 위해 각 단어에서만 pair를 merge한다.

최소한의 python 구현은 다음과 같으며 실제 구현에서는 모든 pair를 indexing하고, 데이터 구조를 증분하여 효율성을 향상시킨다.

논문에서는 BPE를 적용하는데에 있어서
1. source와 target에 대해 독립적으로 인코딩을 수행하는 방식
2. 두 vocabulary의 합집합에 대해 인코딩을 수행하는 방식(joint BPE)
의 두 가지 방식을 적용한다.


첫번째 방식은 텍스트와 vocabulary size에 대해 더 압축적이고 subword unit이 각각의 언어의 학습데이터에서 실제로 등장했음을 보장할 수 있으며 두번째 방식은 source와 target에서의 분할 일관성을 높일 수 있다.
만약 BPE를 독립적으로 적용하면, 동일한 이름이라도 두 언어에서 서로 다르게 segmentation될 수 있으며, 이는 neural model이 subword unit 간의 대응 관계(mapping)를 학습하기 어렵게 만든다.




## Evaluation

이 논문에서는 두가지의 측면에서 평가를 진행하였다.

1. subword unit 사용이 NMT에서 rare word와 unseen word의 번역 성능을 향상시킬 수 있는가?
2. vocabulary 크기, 텍스트 길이(text size), 그리고 번역 품질 측면에서 어떤 subword segmentation 방식이 가장 효과적인가?

실험에는 WMT 2015 데이터를 사용했다. 영어 - 독일어간 번역에는 약 420만 개의 sentence pair와 약 1억 개의 token이 사용되었고 영어 - 러시아어 번역에는 약 260만 개의 sentence pair와 약 5천만 개의 token이 사용되었다.

평가지표는 BLEU Score, CHRF3 Score, unigram F1 Score를 사용하였다.

> **BLEU** : 기계번역 텍스트 품질 평가 알고리즘  
**CHRF3** : 영어 기반 번역에서 인간 판단과 높은 상관관계를 보이는 character n-gram 기반의 F3 score  
**Unigram F1** : clipped unigram precision과 recall의 조화평균 (rare, unseen word의 번역성능을 평가하기 위해 사용)  

모델은 hidden layer : 1000, embedding layer : 620, Optimizer : Adadelta, minibatch size : 80 로 설정하였고 τ = 30000개의 shortlist vocabulary만 메모리에 유지했다. 또한 훈련 방식에서 각 epoch마다 훈련데이터 섞고 각 네트워크는 7일동안 학습하여 12시간마다 저장된 마지막 4개의 model을 가져와 embedding layer를 고정한 상태로 12시간의 추가학습하였다. gradient clipping은 5.0, 1.0의 두 경우에 대해 독립적으로 학습했으나 single model에서는 보통 1.0에서 더 높은 성능을 보였다.

학습 결과 development set에서 가장 높은 성능을 보인 모델과 8개모델에 대한 앙상블을 기반으로 결과 보고했다.


### subword segmentation 평가

![](https://velog.velcdn.com/images/ek__/post/d14a1ec0-dc81-4368-af27-5d794c505f73/image.png)

**character n-gram 분할** : n에 따라 sequence length와 vocabulary size사이의 trade off를 다르게 할수 있다.
unigram은 open vocabulary를 표현할 수 있지만 sequence가 길어지고 bigram은 training vocabulary만으로 test set의 일부 token을 생성할 수 없다는 한계가 존재한다.

**segmentation 알고리즘** (SMT에서 유용하게 사용) : vocab size는 줄일 수 있지만 unknown word problem해결이 불가하며 back-off dictionary없이 번역한다는 목표를 만족시키지 못한다.

**BPE** : open vocabulary 목표를 충족하고 test set에서도 unknown symbol없이 적용가능하다. 또한 character level과 주요한 차이는 더 간결화된 표현이 sequence를 더 짧게하고 attention model이 variable-length unit 단위에서 동작할 수 있게 한다는 점이다.

### 번역 성능 평가

>**baseline**
WDict : back off dictionary 사용
WUnk : back off dictionary 사용하지 않고 OOV를 UNK토큰으로 처리
**subword기반 모델**
backoff dictionary 사용하지 않음

![](https://velog.velcdn.com/images/ek__/post/b395059d-5a20-4780-b294-29d08eda5ab3/image.png)
![](https://velog.velcdn.com/images/ek__/post/8095e4ae-fcdc-4a28-8d9b-9e15bfa819c6/image.png)

   

**실험 결과**
unigram F1 Score에서 subword 기반 모델은 rare word, unseen word에 대해 좋은 점수를 받았다.
또한 OOV에서는 알파벳이 비슷한 영어 -> 독일어 번역에서는 unknown word를 복사해 사용하는 baseline전략도 효과적이었지만 영어->러시아어와 같이 알파벳이 다른 경우에서는 subword model이 더 좋은 성능을 보였다.
또한 jointBPE 방식이 source와 target의 독립적인 encoding 방식과 bigram방식보다 더 효과적임을 알 수 있었다.

단 논문에서는 각 dev set의 모델 간 1 BLEU지표의 차이를 발견하며 각 모델의 랜덤성을 제어하는 것의 후속 연구가 필요하다 밝혔다.


## Analysis

### Unigram accuracy

![](https://velog.velcdn.com/images/ek__/post/35a6f4e3-bdf0-4933-b579-0d4cc4f34b74/image.png)

학습 데이터의 단어 빈도수를 기준으로 target-side단어의 unigram F1 score를 표현한 그래프이다.(단어 등장 빈도에 따라 얼마나 잘 번역했는지) 단어 빈도가 적을수록 대체적으로 감소하는 모습을 보였지만 OOV의 영역에서 baseline이 급격하게 증가한 이유는 독일어로 번역을 할때 대부분의 OOV가 이름이며 이름은 source단어를 그대로 복사하는 방식이 효과적이었기 때문이다. target 단어가 50만개가 넘어가면 back-off 방식(WDict)이 unknown을 그대로 처리하는 방식(WUnk)보다 높은 성능을 보였으며 그보다도 subword 기반(C2-3/500k)이 더 높은 성능을 보였다. 
특히 back-off dictionary는 이름같은 source 단어를 단순히 복사하는 방식이었다면 subword system은 복합어같은 새로운 단어들을 생산해내고 있다는 뜻이다.

5만개의 단어 구간에서는 대부분의 성능이 비슷했지만 C2-3/500K와 C2-50K 사이에서 흥미로운 차이를 볼 수 있다. 두 모델은 shortlist의 차이만 존재하고 C2-3/500K는 이 구간의 단어를 single unit으로 표현했지만 C2-50K는 subword unit으로 표현했다. 그리고 C2-3/500K는 50만 단어 이후 구간에서 subword unit방식을 사용하며 점수를 회복할 수 있었다. 논문에서는 이 차이를 subword unit이 일반 단어보다 희소성이 적기 때문이라고 했다. 이 때문에 network vocabulary size를 줄일 수 있고 더 많은 단어들을 subword unit으로 표현할수 있어 더 높은 점수가 나왔다고 했다.

>즉 single unit으로 표현하면서 점점 단어의 희소성이 커져 번역을 제대로 수행하지 못함


F1 score 만으로는 시스템의 차이를 정확하게 설명하지 못해 추가분석을 수행하였을 때 결과는 다음과 같다.


**영어 -> 독일어**

| Model | Precision | Recall | 특징 |
| --- | --- | --- | --- |
| WDict | 60.6% | 26.5% | back-off dictionary 사용 |
| C2-50k | 29.1% | 33.0% | 가장 많은 OOV 생성 |
| BPE-60k | 32.4% | 26.6% | source와 target을 독립적으로 학습하여 transliteration/copy 오류 존재 |
| BPE-J90k (Joint BPE) | 38.6% | 29.8% | source-target joint BPE 적용 |
    
    
> precision : 모델이 True라고 예측한 것 중에서 실제 True인 것의 비율  
recall : 실제 True 중에서 모델이 True라고 예측한 비율

> WDict는 OOV를 그대로 복사해서 활용하는 과정이 있어 precision이 높고  
반대로 bigram subword system은 합성어를 잘 예측할 수 있어서 recall이 높은 듯하다.

**영어→러시아어 번역**에서는 복사가 효과적으로 동작하지 못해 음역이 필요했다. 그 결과 WDict의 OOV 성능이 크게 낮고 반면 subword 기반 모델은 precision과 recall 모두 크게 향상되었다.

| Model | Precision | Recall |
| --- | --- | --- |
| WDict | 9.2% | 5.2% |
| BPE-J90k (Joint BPE) | 21.9% | 15.6% |


### Manual Analysis

실제 번역이 되는 예시를 보여준다.

baseline system은 독일어, 러시아어 번역 모두를 실패했는데 단어를 삭제해 버리거나 source단어를 그대로 복사하였다.   

**영어 → 독일어**  
subword system은 과도하게 자르거나(bigram) 형태소 단위를 정확하게 맞추지 못해도 번역에 성공했다.   
ex) research→Fo|rs|ch|un|g, Forschungs|instituten   
Forschungs|instituten <- 이게 더 자연스러움  

반면 asinine → dumm 과 같이 데이터 희소성 때문에 번역을 충분히 학습하지 못한 경우에는 잘못된 번역이 생성되기도 한다.  
ex) asinine situation -> asinin-situation (x)  
올바른 번역 dumme situation  

**영어 → 러시아어**  
이 예시에서는 subword system이 음역을 할 수 있음을 보였지만 오류도 발생했다. 모호한 음역 또는 음역을 mapping하는데에 있어서 source 와 target사이의 불일관성때문에 학습에 어려움을 겪었기 때문이다. 그 결과 BPE보다 jointBPE에서 더 좋은 일관성을 보였다.  

## Conclusion

이 논문의 핵심은 rare word와 unseen word를 subword unit의 sequence로 표현하여 NMT모델이 open vocabulary 를 번역할 수 있음을 보인 것이다. 논문에서는 Byte pair encoding기반 알고리즘을 소개했고 open vocabulary를 가변길이의 subword unit으로 구성된 compact한 단어들로 표현할 수 있게 했다. 그 결과 BPE 와 bigram단위의 분할에서도 성능 향상을 가져왔다.  
이런 subword 기반 모델에서 vocabulary size를 줄이는 것이 성능 향상으로 이어짐을 확인했다. 본 논문에서 vocabulary size는 이전논문과의 비교를 통해 임의적으로 설정했다. 이에 따라 향후 언어쌍과 학습데이터의 크기에 따라 자동적으로 vocabulary size를 결정할 수 있는 연구방향을 제안한다.  
